In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

root = '/content/drive/MyDrive/Colab Notebooks'
data_root = os.path.join(root, 'data')
model_root = os.path.join(root, 'model')

os.environ['TORCH_HOME'] = model_root

In [ ]:
model_name = 'resnet50'

In [ ]:
import os
import xml.etree.ElementTree as ET
from PIL import Image
import torch
from torch.utils.data import Dataset

class PetDetectionDataset(Dataset):
    def __init__(self, root, split='trainval', transforms=None):
        self.root = root
        self.transforms = transforms
        split_file = os.path.join(root, 'annotations', f'{split}.txt')
        with open(split_file) as f:
            lines = f.readlines()
        self.imgs = [line.split()[0] for line in lines]

    def __getitem__(self, idx):
        img_name = self.imgs[idx]
        img_path = os.path.join(self.root, 'images', img_name + '.jpg')
        xml_path = os.path.join(self.root, 'annotations', 'xmls', img_name + '.xml')

        img = Image.open(img_path).convert("RGB")

        # parse bounding box
        tree = ET.parse(xml_path)
        root = tree.getroot()
        boxes = []
        for obj in root.findall('object'):
            bndbox = obj.find('bndbox')
            xmin = int(bndbox.find('xmin').text)
            ymin = int(bndbox.find('ymin').text)
            xmax = int(bndbox.find('xmax').text)
            ymax = int(bndbox.find('ymax').text)
            boxes.append([xmin, ymin, xmax, ymax])

        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.ones((len(boxes),), dtype=torch.int64)
        target = {"boxes": boxes, "labels": labels}

        if self.transforms:
            img = self.transforms(img)
        return img, target

    def __len__(self):
        return len(self.imgs)

In [ ]:
import torchvision
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.models.detection.backbone_utils import resnet_fpn_backbone

BEST_MODEL_PATH = os.path.join(model_root, f'{model_name}_pet_best.pth')

# feature pyramid network: adds a 1x1 conv layer after each stage of the resnet and outputs 5 256 channel feature maps on which proposals extracts features from feature maps selected with their area
backbone = resnet_fpn_backbone('resnet50', weights=None)
state_dict = torch.load(BEST_MODEL_PATH, map_location=torch.device('cpu'))['model_state_dict']
backbone.body.load_state_dict(state_dict, strict=False)
backbone.out_channels = 256

model = FasterRCNN(backbone, num_classes=2)
model = model.cuda() if torch.cuda.is_available() else model

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:135: UserWarning: Using 'backbone_name' as positional parameter(s) is deprecated since 0.13 and may be removed in the future. Please use keyword parameter(s) instead.
  warnings.warn(


In [ ]:
from datetime import datetime

def get_time():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision import datasets, models
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim

optimizer = optim.SGD(
    model.parameters(), lr=0.005,
    momentum=0.9, weight_decay=0.0005
)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

transform = transforms.Compose([transforms.ToTensor()])
train_det = PetDetectionDataset(os.path.join(data_root, 'oxford-iiit-pet'), 'trainval', transform)

def collate_fn(batch):
    return tuple(zip(*batch))

train_det_loader = DataLoader(
    train_det, batch_size=4, shuffle=True,
    collate_fn=collate_fn, num_workers=2
)

print(f"[{get_time()}] Start training")

for epoch in range(5):
    model.train()
    for imgs, targets in train_det_loader:
        imgs = [img for img in imgs] # img.cuda()
        targets = [{k: v for k, v in t.items()} for t in targets] # v.cuda()
        loss_dict = model(imgs, targets)
        losses = sum(loss for loss in loss_dict.values())
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
    print(f"[{get_time()}] Epoch {epoch+1} done, loss={losses.item():.4f}")
    scheduler.step()

torch.save(model.state_dict(), os.path.join(model_root, 'resnet_fasterrcnn_pet.pth'))

FileNotFoundError: Caught FileNotFoundError in DataLoader worker process 1.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/fetch.py", line 54, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "/tmp/ipykernel_1331/3747182269.py", line 24, in __getitem__
    tree = ET.parse(xml_path)
           ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/xml/etree/ElementTree.py", line 1204, in parse
    tree.parse(source, parser)
  File "/usr/lib/python3.12/xml/etree/ElementTree.py", line 558, in parse
    source = open(source, "rb")
             ^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Colab Notebooks/data/o